In [75]:
# ==============================================================================
# CHUNK 1 — Imports, load, format, modality exploration
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import mnlogit
from patsy import dmatrix, cr
import warnings
warnings.filterwarnings("ignore")
import re

# ── Paths ──────────────────────────────────────────────────────────────────────
RUN_LABEL = "s2_balanced"
SCALER    = "minmax"
# MCS       = 3406
# MS        = 34
MCS       = 2271
MS        = 15

BASE_DIR    = f"Results/Regular_clustering/Full_dataset/With_counts/{SCALER}/{RUN_LABEL}"
CSV_PATH    = f"{BASE_DIR}/final_mcs{MCS}_ms{MS}/clustering_{RUN_LABEL}_{SCALER}_mcs{MCS}_ms{MS}_labeled.csv"
OUT_DIR_REG = f"{BASE_DIR}/final_mcs{MCS}_ms{MS}/logistic_regression"
os.makedirs(OUT_DIR_REG, exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────────
df_clust = pd.read_csv(CSV_PATH, low_memory=False)

# ── Cluster labels mapping ─────────────────────────────────────────────────────
# 5 clusters
# CLUSTER_LABELS = {
#     1:  "C1 — UHCD + Hospitalization + heavy workup",
#     0:  "C2 — Hospitalized + full workup",
#     2:  "C3 — Discharged + biology +/- ECG",
#     4:  "C4 — Isolated X-ray +/- CT",
#     3:  "C5 — Minimal consumption", # ref
#     -1: "Outliers",
# }
# CLUSTER_ORDER = [
#     "C1 — UHCD + Hospitalization + heavy workup",
#     "C2 — Hospitalized + full workup",
#     "C3 — Discharged + biology +/- ECG",
#     "C4 — Isolated X-ray +/- CT",
#     "C5 — Minimal consumption", # ref
#     "Outliers",
# ]


# 9 clusters
# 9 clusters
CLUSTER_LABELS = {
    0:  "C1 — UHCD + Hospitalization + heavy workup",   # obs=1, hospi=1, bio=1.56, ekg=0.55
    4:  "C2 — Hospitalized + biology + imaging",         # hospi=1, bio=2.19, ct=0.74, ekg=0.34
    5:  "C3 — Hospitalized + biology",                   # hospi=1, bio=0.40, blood=0.20
    6:  "C4 — Hospitalized + ECG + CT",                  # hospi=1, ct=1.00, ekg=0.30, bio=1.00
    7:  "C5 — Hospitalized + ultrasound + biology",      # hospi=1, echo=0.98, bio=1.00
    8:  "C6 — Hospitalized + ECG",                       # hospi=1, ekg=0.48, bio=1.00
    1:  "C7 — Discharged + biology",                     # hospi=0, blood=1.00, bio=1.00
    3:  "C8 — Isolated X-ray",                           # xray=1.00, imaging=1.11
    2:  "C9 — Minimal consumption",                      # tout à 0 = ref
    -1: "Outliers",
}

CLUSTER_ORDER = [
    "C1 — UHCD + Hospitalization + heavy workup",
    "C2 — Hospitalized + biology + imaging",
    "C3 — Hospitalized + biology",
    "C4 — Hospitalized + ECG + CT",
    "C5 — Hospitalized + ultrasound + biology",
    "C6 — Hospitalized + ECG",
    "C7 — Discharged + biology",
    "C8 — Isolated X-ray",
    "C9 — Minimal consumption", # ref
    "Outliers",
]


# Applique les labels
df_clust["cluster_label"] = df_clust["cluster"].map(CLUSTER_LABELS)
df_clust["cluster_label"] = pd.Categorical(
    df_clust["cluster_label"], categories=CLUSTER_ORDER, ordered=True
)

# ── Feature lists ──────────────────────────────────────────────────────────────
''' group complaint too sparse n some clusters'''
# ── Create grouped complaint category ─────────────────────────────────────────
SYSTEMIC_RARE = [
    "Metabolic_Endocrine",
    "Hematology",
    "Poisoning_Intoxication",
]

df_clust["complaint_category_reg"] = df_clust["complaint_category"].replace(
    {cat: "Metabolic_Hematologic_Toxic" for cat in SYSTEMIC_RARE}
)

print("complaint_category_reg distribution:")
print(df_clust["complaint_category_reg"].value_counts())



REG_FEATURES_CATEG = [
    "sex",
    "transport_grouped",
    "age_group",
    "complaint_category_reg",
    "triage",
]

REG_FEATURES_STATUS = [
    "bp_status",
    "hr_status",
    "temp_status",
    "sat_status",
    "rr_status",
    "o2_flow_status",
    "gcs_status",
    "cap_blood_sugar_status",
    "anisocoria_status",
    "urine_dipstick_clean_status",
    "pain_status",
    "breathalyzer_status",
    "hemocue_status",
]

ALL_REG_FEATURES = REG_FEATURES_CATEG + REG_FEATURES_STATUS

# ── Build regression dataframe ─────────────────────────────────────────────────
cols_needed = ALL_REG_FEATURES + ["cluster", "cluster_label", "age"]
df_reg      = df_clust[[c for c in cols_needed if c in df_clust.columns]].copy()
df_reg      = df_reg.dropna(subset=ALL_REG_FEATURES + ["cluster"])

print(f"Regression dataset: {len(df_reg):,} patients")
print(f"Cluster distribution:\n{df_reg['cluster'].value_counts().sort_index()}")

# ── Set correct dtypes ─────────────────────────────────────────────────────────
df_reg["cluster"]            = df_reg["cluster"].astype(int)
df_reg["triage"]             = df_reg["triage"].astype(float).astype(int).astype(str)
df_reg["age_group"]          = df_reg["age_group"].astype(str)
df_reg["sex"]                = df_reg["sex"].astype(str)
df_reg["transport_grouped"]  = df_reg["transport_grouped"].astype(str)
df_reg["complaint_category_reg"] = df_reg["complaint_category_reg"].astype(str)
for col in REG_FEATURES_STATUS:
    if col in df_reg.columns:
        df_reg[col] = df_reg[col].astype(str)

# ── Explore modalities ─────────────────────────────────────────────────────────
print("\nModalities per variable:\n")
for col in ALL_REG_FEATURES:
    if col in df_reg.columns:
        vals = df_reg[col].value_counts().sort_index()
        print(f"── {col}")
        for v, n in vals.items():
            print(f"   {str(v):<35} n={n:,} ({100*n/len(df_reg):.1f}%)")


complaint_category_reg distribution:
complaint_category_reg
Trauma                                13961
Neurological                          12394
Abdominal_Digestive                    7232
Musculoskeletal                        5581
Cardiovascular                         2826
Respiratory                            2444
General_Deterioration                  2383
Dermatological                         1897
ENT_Ophthalmology_Dental               1818
Medical_Followup                       1724
Psychiatric_Behavioral                 1318
Infectious_Fever                       1005
Urogenital_Renal                        904
Metabolic_Hematologic_Toxic             737
Administrative_Social_Unclassified      557
Name: count, dtype: int64
Regression dataset: 56,781 patients
Cluster distribution:
cluster
-1     4201
 0    10306
 1     4286
 2    12507
 3     5995
 4     2609
 5     3636
 6     4924
 7     3011
 8     5306
Name: count, dtype: int64

Modalities per variable:

── sex
   F    

Le cluster de référence doit être le plus grand ou le plus "basal" cliniquement — par exemple le cluster des patients jeunes, non urgents, faible consommation. Tous les OR s'interprètent par rapport à lui.
Avec N=120 000 tout sera significatif comme pour Cramér's V — donc ici aussi tu regardes la magnitude de l'OR, pas juste le p-value. Un OR de 1.05 n'est pas intéressant même si p < 0.001. Concentre-toi sur les OR > 1.5 ou < 0.67 (effet modéré).

In [76]:
# ==============================================================================
# CHUNK 2 — Reference categories + helper function
# Update REF_CATEGORIES after reading modalities from Chunk 1
# ==============================================================================

REF_CATEGORIES = {
    "cluster"                     : 2, # ← C9 — Minimal consumption
    "sex"                         : "M",
    "transport_grouped"           : "Personal",
    "age_group"                   : "15-30",
    "complaint_category_reg"      : "Trauma",
    "triage"                      : "3",
    "bp_status"                   : "normotension",
    "hr_status"                   : "normocardia",
    "temp_status"                 : "normothermia",
    "sat_status"                  : "normal",
    "rr_status"                   : "normal",
    "o2_flow_status"              : "off",
    "gcs_status"                  : "normal",
    "cap_blood_sugar_status"      : "normoglycemia",
    "anisocoria_status"           : "no",
    "urine_dipstick_clean_status" : "negative",
    "pain_status"                 : "no_pain",
    "breathalyzer_status"         : "negative",
    "hemocue_status"              : "normal",
}

ref_c = REF_CATEGORIES["cluster"]

# ── Helper — format results table ──────────────────────────────────────────────


def format_mnlogit_results(result, ref_cluster):
    rows = []
    conf = result.conf_int()

    for cluster in result.params.columns:
        if cluster == ref_cluster:
            continue

        params     = result.params[cluster]
        pvalues    = result.pvalues[cluster]
        cluster_str = str(cluster)
        conf_clust  = conf.xs(cluster_str)  # DataFrame indexé par variable

        for var in params.index:
            if var == "Intercept":
                continue

            or_val = np.exp(params[var])
            pval   = pvalues[var]

            # Utiliser iloc pour éviter les problèmes de noms tronqués
            try:
                row     = conf_clust.loc[var]
                ci_low  = np.exp(row["lower"])
                ci_high = np.exp(row["upper"])
            except KeyError:
                # Chercher par position si loc échoue
                idx = list(params.index).index(var)
                ci_low  = np.exp(conf_clust.iloc[idx]["lower"])
                ci_high = np.exp(conf_clust.iloc[idx]["upper"])

            clean_var = re.sub(
                r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
                r"\1 = \2", var
            )

            rows.append({
                "cluster_vs_ref": f"{CLUSTER_LABELS.get(cluster, f'C{cluster}')} vs {CLUSTER_LABELS.get(ref_cluster, f'C{ref_cluster}')}",
                "variable"      : clean_var,
                "OR"            : round(or_val,  3),
                "CI_low"        : round(ci_low,  3),
                "CI_high"       : round(ci_high, 3),
                "p_value"       : round(pval,    4),
                "significant"   : pval < 0.05,
            })
    return pd.DataFrame(rows)

print("Reference categories set:")
for k, v in REF_CATEGORIES.items():
    print(f"  {k:<35} ref = {v}")
print("\nHelper function loaded.")

Reference categories set:
  cluster                             ref = 2
  sex                                 ref = M
  transport_grouped                   ref = Personal
  age_group                           ref = 15-30
  complaint_category_reg              ref = Trauma
  triage                              ref = 3
  bp_status                           ref = normotension
  hr_status                           ref = normocardia
  temp_status                         ref = normothermia
  sat_status                          ref = normal
  rr_status                           ref = normal
  o2_flow_status                      ref = off
  gcs_status                          ref = normal
  cap_blood_sugar_status              ref = normoglycemia
  anisocoria_status                   ref = no
  urine_dipstick_clean_status         ref = negative
  pain_status                         ref = no_pain
  breathalyzer_status                 ref = negative
  hemocue_status                      ref = norm

In [77]:
# ==============================================================================
# CHUNK 3 — Splines — age vs cluster membership (MULTINOMIAL, ref = C5)
# ==============================================================================

import matplotlib
from matplotlib.lines import Line2D
matplotlib.rcdefaults()
plt.style.use("default")

OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
os.makedirs(OUT_DIR_SPLINE, exist_ok=True)

ref_c_spline = REF_CATEGORIES["cluster"]   # = 3 = C5
ref_label    = CLUSTER_LABELS[ref_c_spline]

df_spline = df_reg[["age", "cluster"]].dropna()
df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

# ── Cluster 3 (C5) = référence → en premier dans l'encodage ──────────────────
df_spline["cluster"] = pd.Categorical(
    df_spline["cluster"],
    categories=[ref_c_spline] + sorted([c for c in df_spline["cluster"].unique() if c != ref_c_spline])
)

age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")

spline_basis_model = sm.add_constant(spline_basis)
spline_basis_pred  = sm.add_constant(spline_basis_range)

# ── MNLogit ───────────────────────────────────────────────────────────────────
y_cat            = df_spline["cluster"].cat.codes
model            = sm.MNLogit(y_cat, spline_basis_model).fit(method="newton", maxiter=500, disp=False)
# Trier non_ref_clusters selon CLUSTER_ORDER
non_ref_clusters = [
    c for label in CLUSTER_ORDER
    for c, lbl in CLUSTER_LABELS.items()
    if lbl == label and c != ref_c_spline and c in df_spline["cluster"].unique()
]
n_outcomes       = len(non_ref_clusters)
logit_preds      = spline_basis_pred.values @ model.params.values

# ── IC ────────────────────────────────────────────────────────────────────────
cov      = model.cov_params()
n_params = spline_basis_pred.shape[1]
ci_lows, ci_highs = [], []

for k in range(n_outcomes):
    idx   = slice(k * n_params, (k + 1) * n_params)
    cov_k = cov.values[idx, idx]
    grad  = spline_basis_pred.values
    se_k  = np.sqrt((grad @ cov_k @ grad.T).diagonal())
    ci_lows.append(logit_preds[:, k] - 1.96 * se_k)
    ci_highs.append(logit_preds[:, k] + 1.96 * se_k)

# ── FIGURE 1 — Un subplot par cluster non-référence ──────────────────────────
palette = sns.color_palette("tab10", n_outcomes)
n_cols  = min(3, n_outcomes)
n_rows  = (n_outcomes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 6, n_rows * 5), facecolor="white")
fig.patch.set_facecolor("white")
axes = np.array(axes).flatten()

df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
    ax        = axes[i]
    c_label   = CLUSTER_LABELS.get(c, f"Cluster {c}")

    # ── Points bruts cohérents avec MNLogit (c vs ref uniquement) ────────────
    df_pair = df_spline[df_spline["cluster"].isin([c, ref_c_spline])]
    raw = (
        df_pair.groupby("age_bin", observed=True)
        .apply(lambda x: (x["cluster"] == c).mean())
        .reset_index()
    )
    raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
    raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
    raw = raw[(raw[0] > 0) & (raw[0] < 1)]

    ax.scatter(raw["age_mid"], raw["logit_raw"],
               color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
    ax.plot(age_range, logit_preds[:, i],
            color=color, linewidth=2, label="Spline fit")
    ax.fill_between(age_range, ci_lows[i], ci_highs[i],
                    alpha=0.2, color=color, label="95% CI")

    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ax.set_facecolor("white")
    ax.set_title(f"{c_label}\nvs {ref_label} (ref)",
                 fontsize=10, fontweight="bold", color="black")
    ax.set_xlabel("Age", fontsize=10, color="black")
    ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=9, color="black")
    ax.tick_params(colors="black")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

legend_elements = [
    Line2D([0], [0], color="grey",      linewidth=0, marker="o",
           markersize=6, alpha=0.5, label="Raw proportion (logit)"),
    Line2D([0], [0], color="black",     linewidth=2,  label="Spline fit"),
    Line2D([0], [0], color="black",     linewidth=8,  alpha=0.2, label="95% CI"),
    Line2D([0], [0], color="lightgrey", linewidth=1,  linestyle="--", label="Age group boundary"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
fig.suptitle(
    f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
    f"Dashed lines = age_group boundaries (30, 45, 60, 75)",
    fontsize=13, fontweight="bold", color="black",
)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig(os.path.join(OUT_DIR_SPLINE, "spline_age_by_cluster_mnlogit.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: spline_age_by_cluster_mnlogit.png")

# ── FIGURE 2 — Tous les clusters sur un seul graphe ──────────────────────────
fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
ax.set_facecolor("white")
fig.patch.set_facecolor("white")

for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
    c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
    ax.plot(age_range, logit_preds[:, i], color=color, linewidth=2, label=c_label)

for boundary in [30, 45, 60, 75]:
    ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

ymax = ax.get_ylim()[1]
for boundary in [30, 45, 60, 75]:
    ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
            fontsize=8, color="grey", va="top")

ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.set_xlabel("Age (years)", fontsize=12, color="black")
ax.set_ylabel(f"Log-odds vs {ref_label}", fontsize=12, color="black")
ax.set_title(
    f"Spline fit (multinomial) — Log-odds of cluster membership vs {ref_label}\n"
    f"(dashed lines = age_group boundaries: 30, 45, 60, 75)",
    fontsize=13, fontweight="bold", color="black", pad=15,
)
ax.tick_params(colors="black")
ax.legend(title=f"Cluster (vs {ref_label})", bbox_to_anchor=(1.02, 1),
          loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_SPLINE, "spline_age_all_clusters_mnlogit.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: spline_age_all_clusters_mnlogit.png")

Saved: spline_age_by_cluster_mnlogit.png
Saved: spline_age_all_clusters_mnlogit.png


In [78]:
# # ==============================================================================
# # CHUNK 3 — Splines — age vs cluster membership (MULTINOMIAL, ref = cluster 4)
# # ==============================================================================
#
# import matplotlib
# from matplotlib.lines import Line2D
# matplotlib.rcdefaults()
# plt.style.use("default")
#
# OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
# os.makedirs(OUT_DIR_SPLINE, exist_ok=True)
#
# df_spline = df_reg[["age", "cluster"]].dropna()
# df_spline = df_spline[df_spline["age"].between(14, 120)].copy()
#
# # Cluster 4 = référence → on le met en premier dans l'encodage
# df_spline["cluster"] = pd.Categorical(
#     df_spline["cluster"],
#     categories=[4] + sorted([c for c in df_spline["cluster"].unique() if c != 4])
# )
#
# age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)
#
# # Base spline
# spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
# spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")
#
# # Ajout intercept pour MNLogit
# spline_basis_model = sm.add_constant(spline_basis)
# spline_basis_pred  = sm.add_constant(spline_basis_range)
#
# # ==============================================================================
# # Ajustement du modèle MULTINOMIAL
# # ==============================================================================
# y_cat  = df_spline["cluster"].cat.codes   # cluster 4 → code 0 = référence
# model  = sm.MNLogit(y_cat, spline_basis_model).fit(method="newton", maxiter=500, disp=False)
#
# # Clusters non-référence (dans l'ordre des codes 1, 2, ...)
# non_ref_clusters = sorted([c for c in df_spline["cluster"].unique() if c != 4])
# n_outcomes       = len(non_ref_clusters)   # nb de colonnes de params
#
# # Prédictions logit : params shape = (n_spline_cols, n_outcomes)
# # logit_preds shape = (300, n_outcomes)
# logit_preds = spline_basis_pred.values @ model.params.values   # shape (300, n_outcomes)
#
# # IC par outcome
# cov      = model.cov_params()   # shape (n_params * n_outcomes, n_params * n_outcomes)
# n_params = spline_basis_pred.shape[1]
#
# ci_lows  = []
# ci_highs = []
#
# for k in range(n_outcomes):
#     # Extraire le bloc de covariance correspondant à l'outcome k
#     idx     = slice(k * n_params, (k + 1) * n_params)
#     cov_k   = cov.values[idx, idx]
#     grad    = spline_basis_pred.values
#     se_k    = np.sqrt((grad @ cov_k @ grad.T).diagonal())
#     ci_lows.append(logit_preds[:, k] - 1.96 * se_k)
#     ci_highs.append(logit_preds[:, k] + 1.96 * se_k)
#
# # ==============================================================================
# # FIGURE 1 — Un subplot par cluster non-référence
# # ==============================================================================
# palette = sns.color_palette("tab10", n_outcomes)
# n_cols  = min(3, n_outcomes)
# n_rows  = (n_outcomes + n_cols - 1) // n_cols
#
# fig, axes = plt.subplots(
#     n_rows, n_cols,
#     figsize   = (n_cols * 6, n_rows * 5),
#     facecolor = "white",
# )
# fig.patch.set_facecolor("white")
# axes = np.array(axes).flatten()
#
# for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
#     ax = axes[i]
#
#     # Proportions brutes en logit (OvR pour visualisation des points bruts)
#     df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)
#     raw = (
#         df_spline.groupby("age_bin", observed=True)
#         .apply(lambda x: (x["cluster"] == c).mean())
#         .reset_index()
#     )
#     raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
#     raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
#     raw = raw[(raw[0] > 0) & (raw[0] < 1)]   # exclure 0 et 1
#
#     ax.scatter(raw["age_mid"], raw["logit_raw"],
#                color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
#     ax.plot(age_range, logit_preds[:, i],
#             color=color, linewidth=2, label="Spline fit (logit)")
#     ax.fill_between(age_range, ci_lows[i], ci_highs[i],
#                     alpha=0.2, color=color, label="95% CI")
#
#     for boundary in [30, 45, 60, 75]:
#         ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#     ax.set_facecolor("white")
#     ax.set_title(f"Cluster {c} vs Cluster 4 (ref)", fontsize=11, fontweight="bold", color="black")
#     ax.set_xlabel("Age",                             fontsize=10, color="black")
#     ax.set_ylabel("Log-odds vs Cluster 4",           fontsize=10, color="black")
#     ax.tick_params(colors="black")
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#
# for j in range(i + 1, len(axes)):
#     axes[j].set_visible(False)
#
# legend_elements = [
#     Line2D([0], [0], color="grey",     linewidth=0, marker="o",
#            markersize=6, alpha=0.5, label="Raw proportion (logit)"),
#     Line2D([0], [0], color="black",    linewidth=2, label="Spline fit (logit)"),
#     Line2D([0], [0], color="black",    linewidth=8, alpha=0.2, label="95% CI"),
#     Line2D([0], [0], color="lightgrey",linewidth=1, linestyle="--", label="Age group boundary"),
# ]
# fig.legend(
#     handles        = legend_elements,
#     loc            = "lower center",
#     ncol           = 4,
#     fontsize       = 9,
#     frameon        = True,
#     bbox_to_anchor = (0.5, 0),
# )
# fig.suptitle(
#     "Spline fit (multinomial) — Log-odds of cluster membership vs Cluster 4\n"
#     "Dashed lines = age_group boundaries (30, 45, 60, 75)",
#     fontsize=13, fontweight="bold", color="black",
# )
# plt.tight_layout(rect=[0, 0.06, 1, 0.95])
# plt.savefig(
#     os.path.join(OUT_DIR_SPLINE, "spline_age_by_cluster_mnlogit.png"),
#     dpi=200, bbox_inches="tight", facecolor="white"
# )
# plt.close()
# print("Saved: spline_age_by_cluster_mnlogit.png")
#
# # ==============================================================================
# # FIGURE 2 — Tous les clusters sur un seul graphe
# # ==============================================================================
# fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
# ax.set_facecolor("white")
# fig.patch.set_facecolor("white")
#
# for i, (c, color) in enumerate(zip(non_ref_clusters, palette)):
#     ax.plot(age_range, logit_preds[:, i], color=color, linewidth=2, label=f"Cluster {c}")
#
# for boundary in [30, 45, 60, 75]:
#     ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
# ymax = ax.get_ylim()[1]
# for boundary in [30, 45, 60, 75]:
#     ax.text(boundary + 0.5, ymax * 0.98, str(boundary), fontsize=8, color="grey", va="top")
#
# ax.axhline(0, color="black", linewidth=0.8, linestyle=":")   # ligne de référence logit=0
#
# ax.set_xlabel("Age (years)",               fontsize=12, color="black")
# ax.set_ylabel("Log-odds vs Cluster 4",     fontsize=12, color="black")
# ax.set_title(
#     "Spline fit (multinomial) — Log-odds of cluster membership vs Cluster 4 (ref)\n"
#     "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
#     fontsize=13, fontweight="bold", color="black", pad=15,
# )
# ax.tick_params(colors="black")
# ax.legend(
#     title          = "Cluster (vs Cluster 4)",
#     bbox_to_anchor = (1.02, 1),
#     loc            = "upper left",
#     fontsize       = 9,
#     title_fontsize = 10,
#     frameon        = True,
# )
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
#
# plt.savefig(
#     os.path.join(OUT_DIR_SPLINE, "spline_age_all_clusters_mnlogit.png"),
#     dpi=200, bbox_inches="tight", facecolor="white"
# )
# plt.close()
# print("Saved: spline_age_all_clusters_mnlogit.png")

In [79]:
# ==============================================================================
# CHUNK 3bis — Splines — age vs cluster membership  BINARY REGRESSION CLUSTER K VS THE REST
# ==============================================================================

import matplotlib
matplotlib.rcdefaults()
plt.style.use("default")

OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
os.makedirs(OUT_DIR_SPLINE, exist_ok=True)

df_spline = df_reg[["age", "cluster"]].dropna()
df_spline = df_spline[df_spline["age"].between(14, 120)].copy()

# Trier clusters selon CLUSTER_ORDER
clusters = [
    c for label in CLUSTER_ORDER
    for c, lbl in CLUSTER_LABELS.items()
    if lbl == label and c in df_spline["cluster"].unique()
]
age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)

spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")

# ── FIGURE 1 — One subplot per cluster ───────────────────────────────────────
from matplotlib.lines import Line2D

n_cols = min(3, len(clusters))
n_rows = (len(clusters) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 6, n_rows * 5), facecolor="white")
fig.patch.set_facecolor("white")
axes = np.array(axes).flatten()

df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)

for i, c in enumerate(clusters):
    ax      = axes[i]
    c_label = CLUSTER_LABELS.get(c, f"Cluster {c}")
    y_bin   = (df_spline["cluster"] == c).astype(int)
    model   = sm.Logit(y_bin, spline_basis).fit(disp=False)

    logit_pred = spline_basis_range.values @ model.params.values
    cov        = model.cov_params()
    gradient   = np.array(spline_basis_range)
    se         = np.sqrt((gradient @ cov.values @ gradient.T).diagonal())
    ci_low     = logit_pred - 1.96 * se
    ci_high    = logit_pred + 1.96 * se

    raw = (
        df_spline.groupby("age_bin", observed=True)
        .apply(lambda x: (x["cluster"] == c).mean())
        .reset_index()
    )
    raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
    raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))
    raw = raw[(raw[0] > 0) & (raw[0] < 1)]

    ax.scatter(raw["age_mid"], raw["logit_raw"],
               color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
    ax.plot(age_range, logit_pred,
            color="steelblue", linewidth=2, label="Spline fit (logit)")
    ax.fill_between(age_range, ci_low, ci_high,
                    alpha=0.2, color="steelblue", label="95% CI")
    for boundary in [30, 45, 60, 75]:
        ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

    ax.set_title(c_label, fontsize=10, fontweight="bold", color="black")
    ax.set_xlabel("Age",  fontsize=10, color="black")
    ax.set_ylabel("Log-odds (logit)", fontsize=10, color="black")
    ax.tick_params(colors="black")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

legend_elements = [
    Line2D([0], [0], color="grey",      linewidth=0, marker="o",
           markersize=6, alpha=0.5, label="Raw proportion"),
    Line2D([0], [0], color="steelblue", linewidth=2, label="Spline fit"),
    Line2D([0], [0], color="steelblue", linewidth=8, alpha=0.2, label="95% CI"),
    Line2D([0], [0], color="lightgrey", linewidth=1, linestyle="--",
           label="Age group boundary"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0))
fig.suptitle(
    "Spline fit — P(cluster membership | age)\n"
    "Dashed lines = age_group boundaries (30, 45, 60, 75)",
    fontsize=13, fontweight="bold", color="black",
)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig(os.path.join(OUT_DIR_SPLINE, "spline_age_by_cluster.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: spline_age_by_cluster.png")

# ── FIGURE 2 — All clusters on one figure ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
palette = sns.color_palette("tab10", len(clusters))

for c, color in zip(clusters, palette):
    c_label    = CLUSTER_LABELS.get(c, f"Cluster {c}")
    y_bin      = (df_spline["cluster"] == c).astype(int)
    model      = sm.Logit(y_bin, spline_basis).fit(disp=False)
    logit_pred = spline_basis_range.values @ model.params.values
    ax.plot(age_range, logit_pred, color=color, linewidth=2, label=c_label)

for boundary in [30, 45, 60, 75]:
    ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)

ymax = ax.get_ylim()[1]
for boundary in [30, 45, 60, 75]:
    ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
            fontsize=8, color="grey", va="top")

ax.set_xlabel("Age (years)",      fontsize=12, color="black")
ax.set_ylabel("Log-odds (logit)", fontsize=12, color="black")
ax.set_title(
    "Spline fit — Log-odds of cluster membership by age\n"
    "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
    fontsize=13, fontweight="bold", color="black", pad=15,
)
ax.tick_params(colors="black")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1),
          loc="upper left", fontsize=8, title_fontsize=9, frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_SPLINE, "spline_age_all_clusters.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: spline_age_all_clusters.png")

Saved: spline_age_by_cluster.png
Saved: spline_age_all_clusters.png


In [80]:
# # ==============================================================================
# # CHUNK 3bis — Splines — age vs cluster membership  BINARY REGRESSION CLUSTER K VS THE REST
# # Justification for categorical age_group in regression
# # ==============================================================================
#
# import matplotlib
# matplotlib.rcdefaults()
# plt.style.use("default")
#
# OUT_DIR_SPLINE = os.path.join(OUT_DIR_REG, "splines")
# os.makedirs(OUT_DIR_SPLINE, exist_ok=True)
#
# df_spline = df_reg[["age", "cluster"]].dropna()
# df_spline = df_spline[df_spline["age"].between(14, 120)].copy()
#
# clusters  = sorted(df_spline["cluster"].unique())
# age_range = np.linspace(df_spline["age"].min(), df_spline["age"].max(), 300)
#
# spline_basis       = dmatrix("cr(age, df=4) - 1", {"age": df_spline["age"]}, return_type="dataframe")
# spline_basis_range = dmatrix("cr(age, df=4) - 1", {"age": age_range},        return_type="dataframe")
#
# # ==============================================================================
# # FIGURE 1 — One subplot per cluster
# # ==============================================================================
# from matplotlib.lines import Line2D
#
# n_cols = min(3, len(clusters))
# n_rows = (len(clusters) + n_cols - 1) // n_cols
#
# fig, axes = plt.subplots(
#     n_rows, n_cols,
#     figsize     = (n_cols * 6, n_rows * 5),
#     facecolor   = "white",
# )
# fig.patch.set_facecolor("white")
# axes = np.array(axes).flatten()
#
# for i, c in enumerate(clusters):
#     ax    = axes[i]
#     y_bin = (df_spline["cluster"] == c).astype(int)
#     model = sm.Logit(y_bin, spline_basis).fit(disp=False)
#
#     # ── Prédiction sur l'échelle LOGIT ──────────────────────────────
#     logit_pred = spline_basis_range.values @ model.params.values
#     cov        = model.cov_params()
#     gradient   = np.array(spline_basis_range)
#     se         = np.sqrt((gradient @ cov.values @ gradient.T).diagonal())
#     ci_low     = logit_pred - 1.96 * se
#     ci_high    = logit_pred + 1.96 * se
#
#     # ── Proportions brutes → aussi en logit ─────────────────────────
#     df_spline["age_bin"] = pd.cut(df_spline["age"], bins=20)
#     raw = (
#         df_spline.groupby("age_bin", observed=True)
#         .apply(lambda x: (x["cluster"] == c).mean())
#         .reset_index()
#     )
#     raw["age_mid"]   = raw["age_bin"].apply(lambda x: x.mid)
#     raw["logit_raw"] = np.log(raw[0] / (1 - raw[0]))  # logit des proportions brutes
#
#     # ── Plot ─────────────────────────────────────────────────────────
#     ax.scatter(raw["age_mid"], raw["logit_raw"],
#                color="grey", s=20, alpha=0.5, label="Raw proportion (logit)")
#     ax.plot(age_range, logit_pred,
#             color="steelblue", linewidth=2, label="Spline fit (logit)")
#     ax.fill_between(age_range, ci_low, ci_high,
#                     alpha=0.2, color="steelblue", label="95% CI")
#     for boundary in [30, 45, 60, 75]:
#         ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
#     title  = "Outliers" if c == -1 else f"Cluster {c}"
#     ylabel = "Log-odds (logit)"
#
#     ax.set_title(title,   fontsize=11, fontweight="bold", color="black")
#     ax.set_xlabel("Age",  fontsize=10, color="black")
#     ax.set_ylabel(ylabel, fontsize=10, color="black")
#     ax.tick_params(colors="black")
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#
# for j in range(i + 1, len(axes)):
#     axes[j].set_visible(False)
#
# legend_elements = [
#     Line2D([0], [0], color="grey",      linewidth=0, marker="o",
#            markersize=6, alpha=0.5,  label="Raw proportion"),
#     Line2D([0], [0], color="steelblue", linewidth=2,  label="Spline fit"),
#     Line2D([0], [0], color="steelblue", linewidth=8,
#            alpha=0.2,                label="95% CI"),
#     Line2D([0], [0], color="lightgrey", linewidth=1,
#            linestyle="--",           label="Age group boundary"),
# ]
# fig.legend(
#     handles        = legend_elements,
#     loc            = "lower center",
#     ncol           = 4,
#     fontsize       = 9,
#     frameon        = True,
#     bbox_to_anchor = (0.5, 0),
# )
# fig.suptitle(
#     "Spline fit — P(cluster membership | age)\n"
#     "Dashed lines = age_group boundaries (30, 45, 60, 75)",
#     fontsize=13, fontweight="bold", color="black",
# )
# plt.tight_layout(rect=[0, 0.06, 1, 0.95])
# plt.savefig(
#     os.path.join(OUT_DIR_SPLINE, "spline_age_by_cluster.png"),
#     dpi=200, bbox_inches="tight", facecolor="white"
# )
# plt.close()
# print("Saved: spline_age_by_cluster.png")
#
# # ==============================================================================
# # FIGURE 2 — All clusters on one figure
# # ==============================================================================
#
# fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
# ax.set_facecolor("white")
# fig.patch.set_facecolor("white")
# palette = sns.color_palette("tab10", len(clusters))
#
# for c, color in zip(clusters, palette):
#     y_bin  = (df_spline["cluster"] == c).astype(int)
#     model  = sm.Logit(y_bin, spline_basis).fit(disp=False)
#     logit_pred = spline_basis_range.values @ model.params.values  # ← logit
#     label  = "Outliers" if c == -1 else f"Cluster {c}"
#     ax.plot(age_range, logit_pred, color=color, linewidth=2, label=label)  # ← logit
#
# for boundary in [30, 45, 60, 75]:
#     ax.axvline(boundary, color="lightgrey", linestyle="--", linewidth=0.8)
#
# ymax = ax.get_ylim()[1]
# for boundary in [30, 45, 60, 75]:
#     ax.text(boundary + 0.5, ymax * 0.98, str(boundary),
#             fontsize=8, color="grey", va="top")
#
# ax.set_xlabel("Age (years)",                   fontsize=12, color="black")
# ax.set_ylabel("Log-odds (logit)", fontsize=12, color="black")
# ax.set_title(
#     "Spline fit — Log-odds of cluster membership by age\n"
#     "(dashed lines = age_group boundaries: 30, 45, 60, 75)",
#     fontsize=13, fontweight="bold", color="black", pad=15,
# )
# ax.tick_params(colors="black")
# ax.legend(
#     title          = "Cluster",
#     bbox_to_anchor = (1.02, 1),
#     loc            = "upper left",
#     fontsize       = 9,
#     title_fontsize = 10,
#     frameon        = True,
# )
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
#
# plt.savefig(
#     os.path.join(OUT_DIR_SPLINE, "spline_age_all_clusters.png"),
#     dpi=200, bbox_inches="tight", facecolor="white"
# )
# plt.close()
# print("Saved: spline_age_all_clusters.png")


Cluster 4 (brun) — forte décroissance avec l'âge
C'est ton cluster "passage simple / aucun examen". Les jeunes (15-30 ans) y sont massivement représentés (~30%) et cette probabilité chute fortement après 45 ans. Les jeunes adultes consultent aux urgences pour des motifs simples ne nécessitant pas d'explorations.
Cluster 7 (jaune-vert) — décroissance progressive
Surreprésenté chez les jeunes (~20%) et quasi absent après 75 ans. Probablement lié à des motifs traumatologiques ou musculo-squelettiques, typiques des jeunes actifs.
Cluster 5 (rose) — forte croissance avec l'âge
Probabilité qui monte fortement après 60 ans et explose après 75 ans (~30%). C'est ton cluster de prise en charge lourde des patients âgés.
Cluster 3 (violet) — légère croissance
Stable avec une légère augmentation après 60 ans — probablement les bilans cardiovasculaires/neurologiques plus fréquents avec l'âge.
Clusters 0 et 1 (orange, vert) — croissance modérée
Augmentent progressivement avec l'âge, pic vers 70-80 ans puis redescendent légèrement. Bilans biologiques et imagerie plus fréquents chez les patients d'âge moyen à âgés.
Cluster 6 (gris) — en cloche
Pic vers 70-80 ans puis décroissance — profil typique des patients âgés mais pas très vieux.
Cluster 2 (rouge) et Outliers (bleu) — en cloche tardive
Pic vers 70-80 ans, profils atypiques ou complexes plus fréquents chez les personnes âgées.
Cluster 8 (cyan) — décroissance
Surreprésenté chez les jeunes, décroît avec l'âge.

Ce que ça justifie pour ta régression :
Les courbes sont clairement non-linéaires — elles montent, descendent, ont des inflexions — ce qui justifie pleinement age_group en catégoriel plutôt qu'age en continu. Et les inflexions coïncident globalement avec tes frontières à 30, 45, 60 et 75 ans, ce qui valide tes coupures a priori.

In [81]:
# 1. Le nom exact de la colonne
print([c for c in df_reg.columns if "complaint" in c.lower()])

# 2. Les modalités et la ref
print(df_reg["complaint_category_reg"].value_counts())
print("Ref définie :", REF_CATEGORIES.get("complaint_category_reg", "NON DÉFINIE"))

['complaint_category_reg']
complaint_category_reg
Trauma                                13961
Neurological                          12394
Abdominal_Digestive                    7232
Musculoskeletal                        5581
Cardiovascular                         2826
Respiratory                            2444
General_Deterioration                  2383
Dermatological                         1897
ENT_Ophthalmology_Dental               1818
Medical_Followup                       1724
Psychiatric_Behavioral                 1318
Infectious_Fever                       1005
Urogenital_Renal                        904
Metabolic_Hematologic_Toxic             737
Administrative_Social_Unclassified      557
Name: count, dtype: int64
Ref définie : Trauma


In [82]:
# Diagnostic complaint_category_reg
var     = "complaint_category_reg"
ref_val = REF_CATEGORIES.get(var, "Trauma")
formula = f"cluster ~ C({var}, Treatment('{ref_val}'))"
print(f"Formula: {formula}")
print(f"Ref: {ref_val}")
print(f"N lignes sans NA: {df_reg[[var, 'cluster']].dropna().shape[0]}")

for method in ["newton", "bfgs", "lbfgs", "cg"]:
    try:
        m = mnlogit(formula, data=df_reg).fit(
            method=method, maxiter=500, disp=True
        )
        print(f"\n✅ {method} OK")
        print(m.summary())
        break
    except Exception as e:
        print(f"❌ {method} failed: {e}")

Formula: cluster ~ C(complaint_category_reg, Treatment('Trauma'))
Ref: Trauma
N lignes sans NA: 56781
Optimization terminated successfully.
         Current function value: nan
         Iterations 4

✅ newton OK
                          MNLogit Regression Results                          
Dep. Variable:                cluster   No. Observations:                56781
Model:                        MNLogit   Df Residuals:                    56646
Method:                           MLE   Df Model:                          126
Date:                Thu, 07 May 2026   Pseudo R-squ.:                     nan
Time:                        15:15:05   Log-Likelihood:                    nan
converged:                       True   LL-Null:                   -1.2349e+05
Covariance Type:            nonrobust   LLR p-value:                       nan
                                                                           cluster=0       coef    std err          z      P>|z|      [0.025      0.975]
---

In [83]:
# ==============================================================================
# CHUNK 4 — Univariate multinomial logistic regressions
# ==============================================================================

print("\n" + "="*60)
print("UNIVARIATE REGRESSIONS")
print("="*60)

all_uni_res = []

for var in ALL_REG_FEATURES:
    if var not in df_reg.columns:
        print(f"MISSING  {var}")
        continue

    ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
    formula = f"cluster ~ C({var}, Treatment('{ref_val}'))"
    print(f"Trying: {formula}")

    model = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model = mnlogit(formula, data=df_reg).fit(
                method  = method,
                maxiter = 2000,
                gtol    = 1e-5,
                disp    = False,
            )
            if not model.mle_retvals.get("converged", True):
                print(f"WARNING {var} (method={method}): converged=False")
            else:
                print(f"OK  {var} (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model is None:
        print(f"FAILED  {var} — all methods failed")
        continue

    df_res = format_mnlogit_results(model, ref_c)
    df_res["model"]   = "univariate"
    df_res["feature"] = var
    all_uni_res.append(df_res)

# ── Concat + CSV ──────────────────────────────────────────────────────────────
if len(all_uni_res) == 0:
    print("WARNING: aucun modèle n'a convergé")
else:
    df_univariate = pd.concat(all_uni_res, ignore_index=True)
    df_univariate = df_univariate.sort_values(
        ["cluster_vs_ref", "feature"]
    ).reset_index(drop=True)

    df_univariate.to_csv(os.path.join(OUT_DIR_REG, "univariate_results.csv"), index=False)
    print(f"\nSaved: univariate_results.csv ({len(df_univariate)} rows)")

    # ── Forest plot — un plot par cluster_vs_ref ──────────────────────────────
    ref_label       = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")
    clusters_vs_ref = sorted(df_univariate["cluster_vs_ref"].unique())

    for clust_label in clusters_vs_ref:
        df_plot = (
            df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
            .sort_values("OR", ascending=True)
            .reset_index(drop=True)
        )
        n_vars = len(df_plot)
        colors = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]

        fig, ax = plt.subplots(
            figsize   = (10, max(8, n_vars * 0.35)),
            facecolor = "white"
        )
        ax.set_facecolor("white")

        ax.barh(
            df_plot["variable"],
            np.log(df_plot["OR"]),
            xerr   = [
                np.log(df_plot["OR"])      - np.log(df_plot["CI_low"]),
                np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]),
            ],
            color   = colors,
            alpha   = 0.8,
            capsize = 3,
            ecolor  = "grey",
        )
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("log(OR) = β", fontsize=12)
        ax.set_title(
            f"Univariate OR — {clust_label}\n"
            f"(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)\n"
            f"ref = {ref_label}",
            fontsize=11, fontweight="bold", color="black"
        )
        ax.tick_params(axis="y", labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        fname_clean = (
            clust_label
            .replace(" ", "_").replace("—", "-")
            .replace("+", "plus").replace("/", "-")
            .replace("(", "").replace(")", "")
        )
        fname = f"forest_plot_univariate_{fname_clean}.png"
        plt.savefig(os.path.join(OUT_DIR_REG, fname),
                    dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()
        print(f"Saved: {fname}")


UNIVARIATE REGRESSIONS
Trying: cluster ~ C(sex, Treatment('M'))
OK  sex (method=bfgs)
Trying: cluster ~ C(transport_grouped, Treatment('Personal'))
OK  transport_grouped (method=bfgs)
Trying: cluster ~ C(age_group, Treatment('15-30'))
OK  age_group (method=bfgs)
Trying: cluster ~ C(complaint_category_reg, Treatment('Trauma'))
OK  complaint_category_reg (method=bfgs)
Trying: cluster ~ C(triage, Treatment('3'))
OK  triage (method=bfgs)
Trying: cluster ~ C(bp_status, Treatment('normotension'))
OK  bp_status (method=bfgs)
Trying: cluster ~ C(hr_status, Treatment('normocardia'))
OK  hr_status (method=bfgs)
Trying: cluster ~ C(temp_status, Treatment('normothermia'))
OK  temp_status (method=bfgs)
Trying: cluster ~ C(sat_status, Treatment('normal'))
OK  sat_status (method=bfgs)
Trying: cluster ~ C(rr_status, Treatment('normal'))
OK  rr_status (method=bfgs)
Trying: cluster ~ C(o2_flow_status, Treatment('off'))
OK  o2_flow_status (method=bfgs)
Trying: cluster ~ C(gcs_status, Treatment('normal')

In [84]:
# # ==============================================================================
# # CHUNK 4 — Univariate multinomial logistic regressions
# # ==============================================================================
#
# print("\n" + "="*60)
# print("UNIVARIATE REGRESSIONS")
# print("="*60)
#
# all_uni_res = []
#
# for var in ALL_REG_FEATURES:
#     if var not in df_reg.columns:
#         print(f"MISSING  {var}")
#         continue
#
#     ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
#     formula = f"cluster ~ C({var}, Treatment('{ref_val}'))"
#     print(f"Trying: {formula}")
#
#     model = None
#     for method in ["bfgs", "lbfgs", "cg", "newton"]:  # bfgs en premier car newton échoue souvent
#         try:
#             model = mnlogit(formula, data=df_reg).fit(
#                 method  = method,
#                 maxiter = 2000,   # augmenté
#                 gtol    = 1e-5,   # tolérance plus souple
#                 disp    = False,
#             )
#             if not model.mle_retvals.get("converged", True):
#                 print(f"WARNING {var} (method={method}): converged=False, résultats conservés")
#             else:
#                 print(f"OK  {var} (method={method})")
#             break
#         except Exception as e:
#             print(f"  {method} failed: {e}")
#
#     if model is None:
#         print(f"FAILED  {var} — all methods failed")
#         continue
#
#     df_res = format_mnlogit_results(model, ref_c)
#     df_res["model"]   = "univariate"
#     df_res["feature"] = var
#     all_uni_res.append(df_res)
#
# # ── Concat ────────────────────────────────────────────────────────────────────
# if len(all_uni_res) == 0:
#     print("WARNING: aucun modèle n'a convergé")
# else:
#     df_univariate = pd.concat(all_uni_res, ignore_index=True)  # ← bug corrigé
#     df_univariate = df_univariate.sort_values(
#         ["cluster_vs_ref", "feature"]
#     ).reset_index(drop=True)
#
#     df_univariate.to_csv(os.path.join(OUT_DIR_REG, "univariate_results.csv"), index=False)
#     print(f"\nSaved: univariate_results.csv ({len(df_univariate)} rows)")
#
#     # ── Forest plot ───────────────────────────────────────────────────────────
# clusters_vs_ref = df_univariate["cluster_vs_ref"].unique()
# n_plots         = len(clusters_vs_ref)
# n_vars          = df_univariate["variable"].nunique()
#
# fig, axes = plt.subplots(1, n_plots,
#                               figsize=(n_plots * 7, max(8, n_vars * 0.3)),
#                               facecolor="white")
# if n_plots == 1:
#     axes = [axes]
#
# for ax, clust_label in zip(axes, clusters_vs_ref):
#     df_plot = (
#         df_univariate[df_univariate["cluster_vs_ref"] == clust_label]
#         .sort_values("OR", ascending=True)
#     )
#     colors = [
#     "crimson"   if (s and or_val > 1) else
#     "steelblue" if (s and or_val <= 1) else
#     "lightgrey"
#     for s, or_val in zip(df_plot["significant"], df_plot["OR"])
#     ]
#     ax.barh(
#         df_plot["variable"],
#         np.log(df_plot["OR"]),
#         xerr   = [
#             np.log(df_plot["OR"]) - np.log(df_plot["CI_low"]),
#             np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]),
#         ],
#         color   = colors,
#         alpha   = 0.8,
#         capsize = 3,
#         ecolor  = "grey",
#     )
#     ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
#     ax.set_title(clust_label, fontsize=11, fontweight="bold")
#     ax.set_xlabel("log(OR)")
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#
# plt.suptitle(
#     f"Univariate log OR — External variables → Cluster membership\n"
#     f"(red = p<0.05, reference cluster = C{ref_c})",
#     fontsize=13, fontweight="bold", y=1.02,
# )
# plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR_REG, "forest_plot_univariate.png"),
#             dpi=200, bbox_inches="tight", facecolor="white")
# plt.close()
# print("Saved: forest_plot_univariate.png")

In [85]:
# ==============================================================================
# CHUNK 4b — Univariate binary logistic regressions — Outliers (-1) vs rest
# ==============================================================================
import statsmodels.formula.api as smf

print("\n" + "="*60)
print("UNIVARIATE REGRESSIONS — OUTLIERS vs REST")
print("="*60)

# Variable binaire : outlier (-1) = 1, tout le reste = 0
df_reg["is_outlier"] = (df_reg["cluster"] == -1).astype(int)
print(f"Outliers: {df_reg['is_outlier'].sum()} / {len(df_reg)}")

all_uni_res_outlier = []

for var in ALL_REG_FEATURES:
    if var not in df_reg.columns:
        print(f"MISSING  {var}")
        continue

    ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
    formula = f"is_outlier ~ C({var}, Treatment('{ref_val}'))"
    print(f"Trying: {formula}")

    model = None
    for method in ["bfgs", "lbfgs", "cg", "newton"]:
        try:
            model = smf.logit(formula, data=df_reg).fit(
                method  = method,
                maxiter = 2000,
                gtol    = 1e-5,
                disp    = False,
            )
            if not model.mle_retvals.get("converged", True):
                print(f"WARNING {var} (method={method}): converged=False, résultats conservés")
            else:
                print(f"OK  {var} (method={method})")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")

    if model is None:
        print(f"FAILED  {var} — all methods failed")
        continue

    # ── Extraire OR, IC, p-value ──────────────────────────────────────────────
    params = model.params
    conf   = model.conf_int()
    pvals  = model.pvalues
    conf.columns = ["CI_low", "CI_high"]

    rows = []
    for v in params.index:
        if v == "Intercept":
            continue

        clean_var = re.sub(
            r"C\((.+?),\s*Treatment\(.+?\)\)\[T\.(.+?)\]",
            r"\1 = \2", v
        )
        rows.append({
            "variable"   : clean_var,
            "OR"         : round(np.exp(params[v]),       3),
            "CI_low"     : round(np.exp(conf.loc[v, "CI_low"]),  3),
            "CI_high"    : round(np.exp(conf.loc[v, "CI_high"]), 3),
            "p_value"    : round(pvals[v], 4),
            "significant": pvals[v] < 0.05,
            "feature"    : var,
            "model"      : "univariate_outlier",
        })

    all_uni_res_outlier.append(pd.DataFrame(rows))

# ── Concat ────────────────────────────────────────────────────────────────────
if len(all_uni_res_outlier) == 0:
    print("WARNING: aucun modèle n'a convergé")
else:
    df_outlier = pd.concat(all_uni_res_outlier, ignore_index=True)
    df_outlier = df_outlier.sort_values(["feature", "variable"]).reset_index(drop=True)

    df_outlier.to_csv(os.path.join(OUT_DIR_REG, "univariate_outlier_results.csv"), index=False)
    print(f"\nSaved: univariate_outlier_results.csv ({len(df_outlier)} rows)")

# ── Forest plot outliers ──────────────────────────────────────────────────────
n_vars  = len(df_outlier)
df_plot = df_outlier.sort_values("OR", ascending=True).reset_index(drop=True)

# Label = feature + variable pour bien distinguer
df_plot["label"] = df_plot["feature"] + " — " + df_plot["variable"]

colors = [
    "crimson"   if (s and or_val > 1)  else
    "steelblue" if (s and or_val <= 1) else
    "lightgrey"
    for s, or_val in zip(df_plot["significant"], df_plot["OR"])
]

fig, ax = plt.subplots(
    figsize   = (10, max(8, n_vars * 0.35)),
    facecolor = "white"
)
ax.set_facecolor("white")

ax.barh(
    df_plot["label"],           # ← label au lieu de variable
    np.log(df_plot["OR"]),
    xerr   = [
        np.log(df_plot["OR"])      - np.log(df_plot["CI_low"]),
        np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]),
    ],
    color   = colors,
    alpha   = 0.8,
    capsize = 3,
    ecolor  = "grey",
)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("log(OR) = β", fontsize=12)
ax.set_title(
    "Univariate OR — External variables → Outliers vs rest\n"
    "(crimson = sig OR>1 | steelblue = sig OR<1 | grey = ns)",
    fontsize=13, fontweight="bold", color="black"
)
ax.tick_params(axis="y", labelsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR_REG, "forest_plot_univariate_outliers.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: forest_plot_univariate_outliers.png")


UNIVARIATE REGRESSIONS — OUTLIERS vs REST
Outliers: 4201 / 56781
Trying: is_outlier ~ C(sex, Treatment('M'))
OK  sex (method=bfgs)
Trying: is_outlier ~ C(transport_grouped, Treatment('Personal'))
OK  transport_grouped (method=bfgs)
Trying: is_outlier ~ C(age_group, Treatment('15-30'))
OK  age_group (method=bfgs)
Trying: is_outlier ~ C(complaint_category_reg, Treatment('Trauma'))
OK  complaint_category_reg (method=bfgs)
Trying: is_outlier ~ C(triage, Treatment('3'))
OK  triage (method=bfgs)
Trying: is_outlier ~ C(bp_status, Treatment('normotension'))
OK  bp_status (method=bfgs)
Trying: is_outlier ~ C(hr_status, Treatment('normocardia'))
OK  hr_status (method=bfgs)
Trying: is_outlier ~ C(temp_status, Treatment('normothermia'))
OK  temp_status (method=bfgs)
Trying: is_outlier ~ C(sat_status, Treatment('normal'))
OK  sat_status (method=bfgs)
Trying: is_outlier ~ C(rr_status, Treatment('normal'))
OK  rr_status (method=bfgs)
Trying: is_outlier ~ C(o2_flow_status, Treatment('off'))
OK  o2_fl

In [86]:
pd.crosstab(df_reg["triage"], df_reg["cluster"])

cluster,-1,0,1,2,3,4,5,6,7,8
triage,,,,,,,,,,
1,31,86,3,0,1,30,6,5,8,27
2,1289,5141,897,439,214,1123,999,1910,1466,2297
3,1734,4128,1945,2970,1619,1167,1690,2189,1195,2218
4,1016,891,1224,6601,3480,276,850,776,328,732
5,131,60,217,2497,681,13,91,44,14,32


In [87]:
# ==============================================================================
# CHUNK 5 — Multivariate multinomial logistic regression
# ==============================================================================

print("\n" + "="*60)
print("MULTIVARIATE REGRESSION")
print("="*60)

formula_parts = []
for var in ALL_REG_FEATURES:
    if var not in df_reg.columns:
        continue
    ref_val = REF_CATEGORIES.get(var, df_reg[var].mode()[0])
    formula_parts.append(f"C({var}, Treatment('{ref_val}'))")

formula_multi = "cluster ~ " + " + ".join(formula_parts)
print(f"\nFormula:\n{formula_multi}\n")

# ── Fit avec fallback comme l'univarié ────────────────────────────────────────
model_multi = None
for method in ["bfgs", "lbfgs", "cg", "newton"]:
    try:
        model_multi = mnlogit(formula_multi, data=df_reg).fit(
            method  = method,
            maxiter = 2000,
            gtol    = 1e-5,
            disp    = False,
        )
        if not model_multi.mle_retvals.get("converged", True):
            print(f"WARNING (method={method}): converged=False, résultats conservés")
        else:
            print(f"OK multivariate (method={method})")
        break
    except Exception as e:
        print(f"  {method} failed: {e}")

if model_multi is None:
    print("FAILED — all methods failed")
else:
    # ── Extraire résultats via format_mnlogit_results ─────────────────────────
    df_multivariate = format_mnlogit_results(model_multi, ref_c)
    df_multivariate["model"] = "multivariate"

    df_multivariate = df_multivariate.sort_values(
        ["cluster_vs_ref", "variable"]
    ).reset_index(drop=True)

    df_multivariate.to_csv(
        os.path.join(OUT_DIR_REG, "multivariate_results.csv"), index=False
    )
    print(f"\nSaved: multivariate_results.csv ({len(df_multivariate)} rows)")

    # ── Forest plot — une seule colonne par cluster_vs_ref ────────────────────
    clusters_vs_ref = sorted(df_multivariate["cluster_vs_ref"].unique())

    ref_label = CLUSTER_LABELS.get(ref_c, f"C{ref_c}")

    for clust_label in clusters_vs_ref:
        df_plot = (
            df_multivariate[df_multivariate["cluster_vs_ref"] == clust_label]
            .sort_values("OR", ascending=True)
            .reset_index(drop=True)
        )
        n_vars = len(df_plot)
        colors = [
            "crimson"   if (s and or_val > 1)  else
            "steelblue" if (s and or_val <= 1) else
            "lightgrey"
            for s, or_val in zip(df_plot["significant"], df_plot["OR"])
        ]

        fig, ax = plt.subplots(
            figsize   = (10, max(8, n_vars * 0.35)),
            facecolor = "white"
        )
        ax.set_facecolor("white")

        ax.barh(
            df_plot["variable"],
            np.log(df_plot["OR"]),
            xerr   = [
                np.log(df_plot["OR"])      - np.log(df_plot["CI_low"]),
                np.log(df_plot["CI_high"]) - np.log(df_plot["OR"]),
            ],
            color   = colors,
            alpha   = 0.8,
            capsize = 3,
            ecolor  = "grey",
        )
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("log(OR) = β", fontsize=12)
        ax.set_title(
            f"Multivariate OR — {clust_label}\n"
            f"(crimson = significant OR>1 | steelblue = significant OR<1 | grey = ns)\n"
            f"ref = {ref_label}",
            fontsize=11, fontweight="bold", color="black"
        )
        ax.tick_params(axis="y", labelsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        # ── Nom de fichier nettoyé ────────────────────────────────────────────────
        fname_clean = (
            clust_label
            .replace(" ", "_")
            .replace("—", "-")
            .replace("+", "plus")
            .replace("/", "-")
            .replace("(", "").replace(")", "")
        )
        fname = f"forest_plot_multivariate_{fname_clean}.png"
        plt.savefig(
            os.path.join(OUT_DIR_REG, fname),
            dpi=200, bbox_inches="tight", facecolor="white"
        )
        plt.close()
        print(f"Saved: {fname}")


MULTIVARIATE REGRESSION

Formula:
cluster ~ C(sex, Treatment('M')) + C(transport_grouped, Treatment('Personal')) + C(age_group, Treatment('15-30')) + C(complaint_category_reg, Treatment('Trauma')) + C(triage, Treatment('3')) + C(bp_status, Treatment('normotension')) + C(hr_status, Treatment('normocardia')) + C(temp_status, Treatment('normothermia')) + C(sat_status, Treatment('normal')) + C(rr_status, Treatment('normal')) + C(o2_flow_status, Treatment('off')) + C(gcs_status, Treatment('normal')) + C(cap_blood_sugar_status, Treatment('normoglycemia')) + C(anisocoria_status, Treatment('no')) + C(urine_dipstick_clean_status, Treatment('negative')) + C(pain_status, Treatment('no_pain')) + C(breathalyzer_status, Treatment('negative')) + C(hemocue_status, Treatment('normal'))

OK multivariate (method=bfgs)

Saved: multivariate_results.csv (504 rows)
Saved: forest_plot_multivariate_C1_-_UHCD_plus_Hospitalization_plus_heavy_workup_vs_C9_-_Minimal_consumption.png
Saved: forest_plot_multivariate